# Practice: Build the Instrument Busa Built

**Computational Text Analysis Pack · Module 21 — The Lineage**

*Technique:* A complete index verborum and concordance, then a reading from it

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/00-foundations/21b_practice_build_a_concordance.ipynb)

Data: `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## The question

> **Build the tool that founded this field, then use it for something a
> frequency count cannot do.**

Module 21 rebuilt Busa's index at demonstration scale. This one builds a working
concordance over a whole author and uses it to answer a question about sense
rather than frequency.

### How to use this notebook

A worked case, not a tutorial. Run it once for the shape of the argument, then
change the question at the top to one of your own.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
import numpy as np
counts = df_colonial["author"].value_counts()
AUTHOR = counts.index[0]        # <- change this
works = df_colonial[df_colonial["author"] == AUTHOR]
full = " ".join(works["text"].astype(str))
tokens = tokenise(full)
print(f"author: {AUTHOR}")
print(f"{works['work_id'].nunique()} works, {len(tokens):,} tokens, "
      f"{len(set(tokens)):,} distinct forms")

index = {}
for pos, w in enumerate(tokens):
    index.setdefault(w, []).append(pos)
print(f"index verborum built: {len(index):,} entries, "
      f"{sum(len(v) for v in index.values()):,} locations")

In [ ]:
def kwic(word, width=40, limit=14):
    out = []
    for pos in index.get(word, [])[:limit]:
        left = " ".join(tokens[max(0, pos-12):pos])[-width:]
        right = " ".join(tokens[pos+1:pos+13])[:width]
        out.append(f"{left:>{width}}  {word.upper():^14}  {right:<{width}}")
    return out

TERM = "native"          # <- change this
print(f"'{TERM}' occurs {len(index.get(TERM, [])):,} times\n")
for line in kwic(TERM):
    print(line)

## The question a count cannot answer

*How many times does the word appear* is a frequency question. *In what
constructions, and with what force* is a reading question, and the concordance
is what makes it tractable at scale: you sort the lines and the senses separate
themselves.

In [ ]:
# Sort concordance lines by their right context — Busa's card-sorting, in code.
positions = index.get(TERM, [])[:200]
lines = []
for pos in positions:
    right = tokens[pos+1:pos+4]
    lines.append((" ".join(right), pos))
lines.sort()
print(f"the {min(len(lines),25)} most common right contexts for '{TERM}':\n")
right_heads = Counter(tokens[p+1] for p in positions if p+1 < len(tokens))
for w, n in right_heads.most_common(12):
    print(f"   {TERM} {w:<18}{n:>5}")
print()
print("That table is a sense inventory in miniature. Words following the node")
print("cluster into the constructions the author actually used — which is a")
print("different object from the collocation list module 5 produces, because it")
print("is positional rather than windowed.")

In [ ]:
# Dispersion: sorted cards, plotted. Where in the author's output does it fall?
plt.figure(figsize=(9, 2.4))
plt.scatter(index.get(TERM, []), [0]*len(index.get(TERM, [])), s=5, alpha=.5)
plt.yticks([])
plt.xlabel(f"token position across {AUTHOR}'s works")
plt.title(f"Dispersion of '{TERM}'")
plt.tight_layout()

n_parts = 10
size = max(len(tokens)//n_parts, 1)
per = [sum(1 for p in index.get(TERM, []) if i*size <= p < (i+1)*size)
       for i in range(n_parts)]
print("occurrences per tenth of the corpus:", per)
print()
print("Even spread means the word belongs to the author's habitual vocabulary.")
print("Clumping means it belongs to one work or one episode — and module 10d's")
print("DP is how you turn that impression into a number.")

## What this licenses

**Defensible:** "Across [author]'s N works, '[term]' occurs M times; the most
frequent following words are [list]; occurrences are distributed [evenly /
concentrated in work X]."

**Not defensible:** a claim about what the author *meant* by the term, from the
count alone. The concordance supports the claim; it does not make it. Busa's
project produced an index precisely so that reading Aquinas could be exhaustive
rather than selective — the instrument serves the reading, not the reverse.

**And a limit:** this corpus is passages sampled from works, not the complete
works. Module 22c's audit applies: check what your index is an index *of*.

### 🔧 Your turn 1

Pick a term whose sense you expect to have shifted, and sort the concordance by LEFT context instead. Does a different pattern appear?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Left-sorting groups by what precedes the node, which surfaces determiners,
modifiers and grammatical frames rather than the node's objects — often a
cleaner separation of senses than right-sorting gives. Concordancers have
offered both since the 1960s for exactly this reason, and choosing the sort is a
research decision worth stating. It is also the cheapest way to discover that a
word you were treating as one item is doing two jobs.

</details>